# US 3.4 -- Translation Quality Evaluation

After training the CycleGAN (baseline or defect-aware), we need to measure
how good the translations are.  This notebook covers the quantitative and
qualitative evaluation methods.

## What you will learn

1. Structural Similarity Index (SSIM) for translation quality
2. Defect Dice score for defect preservation
3. Visual side-by-side comparison
4. The `udm-epic3 evaluate` CLI command

## Prerequisites

- Python 3.12, PyTorch
- Install: `uv pip install -e ".[epic3]"`
- Trained CycleGAN checkpoint (see `epic3_02_training.ipynb`)
- Paired evaluation data (see `epic3_01_data_prep.ipynb`)

---
## 1. Evaluation Metrics Overview

We use two complementary metrics:

| Metric | What it measures | Range | Better |
|--------|-----------------|-------|--------|
| **SSIM** | Structural similarity between translated and real USM | [0, 1] | Higher |
| **Defect Dice** | Overlap of defect masks before and after translation | [0, 1] | Higher |

SSIM requires **paired** data (same subject imaged in both modalities).
Defect Dice requires **defect masks** for the AOI images.

### Why SSIM?

SSIM measures perceptual similarity by comparing luminance, contrast, and
structure.  It is more aligned with human perception than pixel-level L1/L2
distance.  A high SSIM between translated AOI and real USM indicates that
the generator produces realistic USM-like images.

### Why Defect Dice?

The whole point of translation is to preserve labels.  Defect Dice directly
measures whether the defect regions survive the translation process.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from udm_epic3.evaluation.quality_metrics import (
    compute_ssim,
    compute_defect_dice,
    evaluate_translation,
)

---
## 2. SSIM Computation

SSIM is computed on paired data: the translated AOI (fake USM) vs the real
USM image of the same subject.

In [ ]:
# Simulate paired evaluation data
# In practice, these come from the PairedDataset and the trained generator
translated_usm = torch.randn(4, 3, 256, 256)  # G_AB(aoi_images)
real_usm = torch.randn(4, 3, 256, 256)         # ground-truth USM images

# Compute SSIM for the batch
ssim_scores = compute_ssim(translated_usm, real_usm)

print(f"SSIM scores per image: {ssim_scores}")
print(f"Mean SSIM: {np.mean(ssim_scores):.4f}")
print(f"\nNote: random data gives low SSIM (~0.0).")
print(f"A trained CycleGAN typically achieves SSIM > 0.5 on paired data.")

---
## 3. Defect Dice Score

The Defect Dice score measures how well defect regions are preserved after
translation.  We compare the original AOI defect mask with the mask
extracted from the translated image.

In [ ]:
# Simulate defect masks
# original_mask: known defect mask from AOI labels
original_mask = torch.zeros(4, 1, 256, 256)
original_mask[:, :, 100:140, 100:140] = 1.0  # simulate defect region

# translated_mask: defect mask extracted from translated USM image
# (slightly shifted to simulate imperfect preservation)
translated_mask = torch.zeros(4, 1, 256, 256)
translated_mask[:, :, 102:142, 98:138] = 1.0  # slightly offset

# Compute Dice score
dice_scores = compute_defect_dice(original_mask, translated_mask)

print(f"Defect Dice scores per image: {dice_scores}")
print(f"Mean Defect Dice: {np.mean(dice_scores):.4f}")
print(f"\nA score of 1.0 means perfect defect preservation.")
print(f"The slight offset gives < 1.0 due to boundary mismatch.")

---
## 4. Visual Side-by-Side Comparison

Numbers alone do not tell the whole story.  Visual inspection is essential
for evaluating translation quality.

In [ ]:
def side_by_side_comparison(real_A, fake_B, real_B, n_samples=4):
    """Show AOI, translated USM, and real USM side by side."""
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    
    if n_samples == 1:
        axes = axes[np.newaxis, :]
    
    for i in range(n_samples):
        # Denormalize from [-1, 1] to [0, 1]
        img_a = (real_A[i] * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
        img_fake = (fake_B[i] * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
        img_b = (real_B[i] * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()
        
        axes[i, 0].imshow(img_a)
        axes[i, 0].set_title("Original AOI" if i == 0 else "")
        axes[i, 0].axis("off")
        
        axes[i, 1].imshow(img_fake)
        axes[i, 1].set_title("Translated USM" if i == 0 else "")
        axes[i, 1].axis("off")
        
        axes[i, 2].imshow(img_b)
        axes[i, 2].set_title("Real USM" if i == 0 else "")
        axes[i, 2].axis("off")
    
    fig.suptitle("Translation Quality: AOI -> USM", fontsize=16)
    fig.tight_layout()
    plt.show()

# Demo with random data (replace with real model outputs)
real_A = torch.randn(4, 3, 256, 256)
fake_B = torch.randn(4, 3, 256, 256)
real_B = torch.randn(4, 3, 256, 256)

side_by_side_comparison(real_A, fake_B, real_B, n_samples=4)

---
## 5. Full Evaluation Pipeline

The `evaluate_translation` function runs the complete evaluation on a paired
dataset and returns a summary report.

In [ ]:
# NOTE: In practice, pass a trained model and paired dataset.
# This shows the API and expected output format.
#
# results = evaluate_translation(
#     model=model,
#     paired_dataset=paired_dataset,
#     device="cuda",
# )
#
# Expected output:
# {
#     'ssim_mean': 0.72,
#     'ssim_std': 0.08,
#     'defect_dice_mean': 0.85,
#     'defect_dice_std': 0.06,
#     'n_samples': 50,
#     'per_sample': [...]  # per-sample scores
# }

print("evaluate_translation() returns a dict with:")
print("  - ssim_mean / ssim_std: overall structural similarity")
print("  - defect_dice_mean / defect_dice_std: defect preservation")
print("  - n_samples: number of paired samples evaluated")
print("  - per_sample: list of per-sample metric dicts")

---
## 6. CLI: `udm-epic3 evaluate`

Run the full evaluation from the command line:

```bash
# Evaluate a trained CycleGAN checkpoint
udm-epic3 evaluate \
    --config configs/epic3_cyclegan.yaml \
    --checkpoint outputs/epic3/checkpoints/epoch_200.pt
```

This will:
1. Load the paired evaluation dataset
2. Translate all AOI images to USM using the trained generator
3. Compute SSIM against real USM images
4. Compute Defect Dice against original masks
5. Save visual comparisons to `outputs/epic3/evaluation/`
6. Print a summary table:

```
=== Translation Quality Report ===
Metric              Mean     Std
SSIM                0.72     0.08
Defect Dice         0.85     0.06
Samples evaluated   50
```

---
## Next steps

| Notebook | Topic |
|----------|------|
| `epic3_05_downstream.ipynb` | Train segmentation on translated data |
| `epic3_06_multisite.ipynb` | Multi-site generalization |